In [ ]:
# Global settings: edit this cell, then run all cells.
ENVIRONMENT = "hopper"  # halfcheetah, hopper, or walker (walker2d also accepted)
DATASET = "medium-replay-v2"
CORRUPTION = "clean"  # clean, random, or adversarial
CORRUPTION_TARGET = "none"  # none, observations, actions, rewards, dynamics, or mixed

TRAINING_STARTED_AT = None  # e.g. "2026-08-08 01:19:52"; None selects newest.
INCLUDE_RUNNING = True  # Show metrics from the algorithm that is currently training.
SMOOTHING_WINDOW = 1  # 1 disables rolling smoothing.

# Corruption-robust offline-to-online comparison

The newest comparison matching the global settings is discovered automatically. Available completed algorithms and the currently running algorithm are included.

In [ ]:
import json
import re
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold"})
pd.set_option("display.max_columns", 30)

PAPER_ORDER = [
    "rpex",
    "riql_pex",
    "riql_naive",
    "uwmsg",
    "pex",
    "cal_ql",
    "wsrl",
    "ro2o",
    "pessimistic_q_ensemble",
]
DISPLAY_NAMES = {
    "rpex": "RPEX",
    "riql_pex": "RIQL+PEX",
    "riql_naive": "RIQL naive",
    "uwmsg": "UWMSG",
    "pex": "PEX",
    "cal_ql": "Cal-QL",
    "wsrl": "WSRL",
    "ro2o": "RO2O",
    "pessimistic_q_ensemble": "Pessimistic Q-Ensemble",
}
COLORS = dict(zip(PAPER_ORDER, plt.cm.tab10.colors[: len(PAPER_ORDER)]))
ENVIRONMENT_ALIASES = {
    "halfcheetah": "halfcheetah",
    "half-cheetah": "halfcheetah",
    "hopper": "hopper",
    "walker": "walker2d",
    "walker2d": "walker2d",
}
VALID_CORRUPTIONS = {"clean", "random", "adversarial"}
VALID_TARGETS = {"none", "observations", "actions", "rewards", "dynamics", "mixed"}
REQUIRED_COLUMNS = {
    "phase",
    "step",
    "env_steps",
    "elapsed_seconds",
    "return_mean",
    "normalized_return_mean",
    "normalized_return_std",
}


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "robust_o2o").is_dir() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not locate the corruption_robust_o2o project root")


environment_key = ENVIRONMENT.strip().lower()
if environment_key not in ENVIRONMENT_ALIASES:
    raise ValueError(f"ENVIRONMENT must be one of {sorted(ENVIRONMENT_ALIASES)}")
if CORRUPTION not in VALID_CORRUPTIONS:
    raise ValueError(f"CORRUPTION must be one of {sorted(VALID_CORRUPTIONS)}")
if CORRUPTION_TARGET not in VALID_TARGETS:
    raise ValueError(f"CORRUPTION_TARGET must be one of {sorted(VALID_TARGETS)}")

DOMAIN = ENVIRONMENT_ALIASES[environment_key]
ENV_NAME = f"{DOMAIN}-{DATASET}"
TARGET = "none" if CORRUPTION == "clean" else CORRUPTION_TARGET
if CORRUPTION != "clean" and TARGET == "none":
    raise ValueError("random/adversarial corruption requires a non-none target")

PROJECT_ROOT = find_project_root()
RESULTS_ROOT = PROJECT_ROOT / "results"
SETTING_DIR = RESULTS_ROOT / "comparisons" / ENV_NAME / CORRUPTION / TARGET
if not SETTING_DIR.is_dir():
    raise FileNotFoundError(f"No result setting directory: {SETTING_DIR}")
comparison_candidates = [
    path for path in SETTING_DIR.iterdir()
    if path.is_dir() and (path / "runs").is_dir()
]
if not comparison_candidates:
    raise FileNotFoundError(f"No comparison found under {SETTING_DIR}")


def comparison_started_at(path: Path) -> datetime:
    manifest_path = path / "manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("start_time"):
            return datetime.fromisoformat(manifest["start_time"])
    match = re.search(r"(\d{8}_\d{6})", path.name)
    if match:
        return datetime.strptime(match.group(1), "%Y%m%d_%H%M%S")
    return datetime.fromtimestamp(path.stat().st_mtime)


if TRAINING_STARTED_AT is None:
    COMPARISON_DIR = max(comparison_candidates, key=comparison_started_at)
else:
    requested_start = pd.Timestamp(TRAINING_STARTED_AT).to_pydatetime().replace(tzinfo=None)
    matches = [
        path for path in comparison_candidates
        if comparison_started_at(path).replace(tzinfo=None) == requested_start
    ]
    if not matches:
        available = "\n".join(
            f"  - {comparison_started_at(path)} | {path.name}"
            for path in sorted(comparison_candidates, key=comparison_started_at)
        )
        raise FileNotFoundError(
            f"No comparison started at {TRAINING_STARTED_AT!r}. Available:\n{available}"
        )
    COMPARISON_DIR = matches[-1]
RUNS_DIR = COMPARISON_DIR / "runs"


def load_metrics(runs_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    frames: list[pd.DataFrame] = []
    run_records: list[dict] = []
    for metrics_path in sorted(runs_dir.rglob("metrics.csv")):
        run_dir = metrics_path.parent
        config_path = run_dir / "config.json"
        summary_path = run_dir / "summary.json"
        if not config_path.exists():
            continue
        config = json.loads(config_path.read_text(encoding="utf-8"))
        algorithm = config.get("algorithm")
        if algorithm not in PAPER_ORDER:
            continue
        summary = (
            json.loads(summary_path.read_text(encoding="utf-8"))
            if summary_path.exists()
            else {}
        )
        status = summary.get("status", "running")
        if status == "failed" or (status != "completed" and not INCLUDE_RUNNING):
            continue
        try:
            frame = pd.read_csv(metrics_path)
        except (OSError, pd.errors.ParserError) as exc:
            print(f"Skipping metrics while it is being written: {metrics_path} ({exc})")
            continue
        missing = REQUIRED_COLUMNS - set(frame.columns)
        if frame.empty or missing:
            continue
        run_key = str(run_dir.resolve())
        seed = int(config.get("seed", -1))
        frame = frame.copy()
        frame["metric_order"] = np.arange(len(frame))
        frame["algorithm"] = algorithm
        frame["algorithm_name"] = DISPLAY_NAMES[algorithm]
        frame["seed"] = seed
        frame["run_dir"] = run_key
        frame["run_status"] = status
        frame["protocol"] = config.get("protocol", "unspecified")
        frame["planned_offline_steps"] = int(config["offline_steps"])
        frame["planned_online_steps"] = int(config["online_steps"])
        frames.append(frame)
        run_records.append(
            {
                "algorithm": algorithm,
                "algorithm_name": DISPLAY_NAMES[algorithm],
                "seed": seed,
                "run_dir": run_key,
                "status": status,
                "protocol": config.get("protocol", "unspecified"),
                "planned_offline_steps": int(config["offline_steps"]),
                "planned_online_steps": int(config["online_steps"]),
                "run_elapsed_seconds": summary.get("elapsed_seconds", np.nan),
            }
        )
    if not frames:
        raise RuntimeError(f"No usable metrics.csv files found below {runs_dir}")

    metrics = pd.concat(frames, ignore_index=True)
    runs = pd.DataFrame(run_records)
    latest = (
        runs.sort_values("run_dir")
        .drop_duplicates(["algorithm", "seed"], keep="last")
        ["run_dir"]
    )
    runs = runs[runs["run_dir"].isin(latest)].copy()
    metrics = metrics[metrics["run_dir"].isin(latest)].copy()
    category = pd.CategoricalDtype(PAPER_ORDER, ordered=True)
    runs["algorithm"] = runs["algorithm"].astype(category)
    metrics["algorithm"] = metrics["algorithm"].astype(category)
    metrics = metrics.sort_values(["algorithm", "seed", "metric_order"])
    runs = runs.sort_values(["algorithm", "seed"])
    return metrics, runs


metrics, runs = load_metrics(RUNS_DIR)
OFFLINE_TARGET = int(runs["planned_offline_steps"].max())
ONLINE_TARGET = int(runs["planned_online_steps"].max())
if runs["planned_offline_steps"].nunique() > 1 or runs["planned_online_steps"].nunique() > 1:
    print("Warning: selected runs use different step budgets; plot limits use the maximum values.")
print(f"Comparison: {COMPARISON_DIR}")
print(f"Started:    {comparison_started_at(COMPARISON_DIR)}")
print(f"Protocols:  {', '.join(sorted(runs['protocol'].unique()))}")
print(f"Setting:    {ENV_NAME} | {CORRUPTION} | {TARGET}")
print(f"Schedule:   offline={OFFLINE_TARGET:,}, online={ONLINE_TARGET:,}")
print(f"Available:  {', '.join(str(value) for value in metrics['algorithm'].dropna().unique())}")

## Auto-refreshed plots

In [ ]:
from IPython.display import Image

for title, filename in (
    ("Offline → online", "comparison_offline_online.png"),
    ("Offline", "comparison_offline.png"),
    ("Online", "comparison_online.png"),
):
    path = COMPARISON_DIR / filename
    print(f"{title}: {path}")
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print("  Plot has not been generated yet.")

## 1. Table

In [ ]:
run_rows = []
for run in runs.itertuples(index=False):
    frame = metrics[metrics["run_dir"] == run.run_dir].sort_values("metric_order")
    offline = frame[frame["phase"] == "offline"]
    online = frame[frame["phase"] == "online"]
    latest = frame.iloc[-1]
    elapsed_seconds = run.run_elapsed_seconds
    if pd.isna(elapsed_seconds):
        elapsed_seconds = float(frame["elapsed_seconds"].max())
    run_rows.append(
        {
            "algorithm": str(run.algorithm),
            "algorithm_name": run.algorithm_name,
            "seed": run.seed,
            "status": run.status,
            "offline_updates": int(offline["step"].max()) if not offline.empty else 0,
            "offline_target": run.planned_offline_steps,
            "online_steps": int(online["env_steps"].max()) if not online.empty else 0,
            "online_target": run.planned_online_steps,
            "latest_normalized_return": float(latest["normalized_return_mean"]),
            "best_normalized_return": float(frame["normalized_return_mean"].max()),
            "elapsed_hours": float(elapsed_seconds) / 3600.0,
        }
    )
run_table = pd.DataFrame(run_rows)


def merge_status(values: pd.Series) -> str:
    statuses = set(values)
    if statuses == {"completed"}:
        return "completed"
    if "running" in statuses:
        return "running"
    return "mixed"


table = (
    run_table.groupby(["algorithm", "algorithm_name"], observed=True)
    .agg(
        status=("status", merge_status),
        seeds=("seed", "nunique"),
        offline_updates=("offline_updates", "max"),
        offline_target=("offline_target", "max"),
        online_steps=("online_steps", "max"),
        online_target=("online_target", "max"),
        latest_normalized_return=("latest_normalized_return", "mean"),
        best_normalized_return=("best_normalized_return", "mean"),
        elapsed_hours=("elapsed_hours", "mean"),
    )
    .reset_index()
)
order = {algorithm: index for index, algorithm in enumerate(PAPER_ORDER)}
table["order"] = table["algorithm"].map(order)
table = table.sort_values("order").drop(columns="order").reset_index(drop=True)
table["offline progress"] = table.apply(
    lambda row: f"{int(row.offline_updates):,} / {int(row.offline_target):,}", axis=1
)
table["online progress"] = table.apply(
    lambda row: f"{int(row.online_steps):,} / {int(row.online_target):,}", axis=1
)
display_table = table[
    [
        "algorithm_name",
        "status",
        "seeds",
        "offline progress",
        "online progress",
        "latest_normalized_return",
        "best_normalized_return",
        "elapsed_hours",
    ]
].rename(
    columns={
        "algorithm_name": "Algorithm",
        "status": "Status",
        "seeds": "Seeds",
        "latest_normalized_return": "Latest normalized return",
        "best_normalized_return": "Best normalized return",
        "elapsed_hours": "Elapsed hours",
    }
)
display(
    display_table.style.format(
        {
            "Latest normalized return": "{:.2f}",
            "Best normalized return": "{:.2f}",
            "Elapsed hours": "{:.2f}",
        }
    )
)

## 2. Offline → online

In [ ]:
def aggregate_phase(phase: str) -> pd.DataFrame:
    x_column = "env_steps" if phase == "online" else "step"
    phase_metrics = metrics[metrics["phase"] == phase]
    curve = (
        phase_metrics.groupby(["algorithm", x_column], observed=True)
        .agg(
            mean=("normalized_return_mean", "mean"),
            seed_std=("normalized_return_mean", "std"),
            episode_std=("normalized_return_std", "mean"),
            seeds=("seed", "nunique"),
        )
        .reset_index()
        .sort_values(["algorithm", x_column])
    )
    curve["band"] = np.where(
        curve["seeds"] > 1, curve["seed_std"], curve["episode_std"]
    )
    curve["band"] = curve["band"].fillna(0.0)
    if SMOOTHING_WINDOW > 1:
        for column in ("mean", "band"):
            curve[column] = curve.groupby("algorithm", observed=True)[column].transform(
                lambda values: values.rolling(SMOOTHING_WINDOW, min_periods=1).mean()
            )
    return curve


def draw_curve(axis: plt.Axes, curve: pd.DataFrame, x_column: str) -> None:
    for algorithm in PAPER_ORDER:
        part = curve[curve["algorithm"] == algorithm]
        if part.empty:
            continue
        x = part[x_column].to_numpy(dtype=float)
        mean = part["mean"].to_numpy(dtype=float)
        band = part["band"].to_numpy(dtype=float)
        axis.plot(x, mean, color=COLORS[algorithm], linewidth=1.9, label=DISPLAY_NAMES[algorithm])
        axis.fill_between(x, mean - band, mean + band, color=COLORS[algorithm], alpha=0.10)


offline_curve = aggregate_phase("offline").rename(columns={"step": "phase_step"})
offline_curve["continuous_step"] = offline_curve["phase_step"]
online_curve = aggregate_phase("online").rename(columns={"env_steps": "phase_step"})
online_curve["continuous_step"] = OFFLINE_TARGET + online_curve["phase_step"]
continuous_curve = pd.concat([offline_curve, online_curve], ignore_index=True).sort_values(
    ["algorithm", "continuous_step"]
)

figure, axis = plt.subplots(figsize=(15, 6.5))
draw_curve(axis, continuous_curve, "continuous_step")
axis.axvline(OFFLINE_TARGET, color="black", linestyle="--", linewidth=1.5, label="Offline → online")
axis.axvspan(OFFLINE_TARGET, OFFLINE_TARGET + ONLINE_TARGET, color="tab:green", alpha=0.035)
axis.set_xlim(0, OFFLINE_TARGET + ONLINE_TARGET)
axis.set_xlabel("Offline updates + online environment steps")
axis.set_ylabel("D4RL normalized return")
axis.set_title(f"Offline → online | {ENV_NAME} | {CORRUPTION} {TARGET}")
axis.xaxis.set_major_formatter(lambda value, _position: f"{value / 1e6:g}M")
axis.grid(alpha=0.25)
axis.legend(ncol=5, fontsize=9, loc="lower center", bbox_to_anchor=(0.5, -0.27))
figure.tight_layout()
plt.show()

## 3. Offline

In [ ]:
figure, axis = plt.subplots(figsize=(15, 6.5))
draw_curve(axis, aggregate_phase("offline"), "step")
axis.set_xlim(0, OFFLINE_TARGET)
axis.set_xlabel("Offline gradient updates")
axis.set_ylabel("D4RL normalized return")
axis.set_title(f"Offline | {ENV_NAME} | {CORRUPTION} {TARGET}")
axis.xaxis.set_major_formatter(lambda value, _position: f"{value / 1e6:g}M")
axis.grid(alpha=0.25)
axis.legend(ncol=5, fontsize=9, loc="lower center", bbox_to_anchor=(0.5, -0.27))
figure.tight_layout()
plt.show()

## 4. Online

In [ ]:
figure, axis = plt.subplots(figsize=(15, 6.5))
current_online_curve = aggregate_phase("online")
if current_online_curve.empty:
    axis.text(0.5, 0.5, "No online metrics yet", transform=axis.transAxes, ha="center", va="center")
else:
    draw_curve(axis, current_online_curve, "env_steps")
axis.set_xlim(0, ONLINE_TARGET)
axis.set_xlabel("Online environment steps")
axis.set_ylabel("D4RL normalized return")
axis.set_title(f"Online | {ENV_NAME} | {CORRUPTION} {TARGET}")
axis.xaxis.set_major_formatter(lambda value, _position: f"{value / 1e6:g}M")
axis.grid(alpha=0.25)
if not current_online_curve.empty:
    axis.legend(ncol=5, fontsize=9, loc="lower center", bbox_to_anchor=(0.5, -0.27))
figure.tight_layout()
plt.show()